# Functional Closures and Decorators

##  Closures
* In order to understand closures, let's be sure we understand Python's scoping rules: LEGB
  * L = local
  * E = enclosing
  * G = global
  * B = builtin (e.g., __`len()`__ function)

In [ ]:
global_var = 'global var'
   
def outer_func():
    enclosing_var = 'local to outer_func()'
    
    def inner_func():
        local_var = 'local to inner_func()'
        print(local_var)
        print(enclosing_var, 'enclosing scope')
        print(global_var, 'global scope')
        
    inner_func()

In [ ]:
outer_func()

* When a function references a name that is not local, Python first attempts to resolve that name in the enclosing scope
* A *closure* is a nested function that remembers a value or values from the enclosing lexical scope even when the program flow is no longer in the enclosing scope
* Why would we want a function that remembers a value?
  * lightweight state, i.e., to remember a bit of information
  * small helper function (alternative to lambda when logic gets more complex)
  * you don't want the overhead of a class

In [ ]:
def make_counter():
    count = 0 # this variable in the enclosing scope is important
    print(f'id(count): 0x{id(count):x}') 

    def increment():
        nonlocal count # get write access to outer count var
        count += 1
        
        return count
    
    print(f'id(increment): 0x{id(increment):x}')
    
    return increment

In [ ]:
response_counter = make_counter()

while (response := input('Enter something (RETURN to end): ')):
    print(f'Response number {response_counter()} was {response}')

In [ ]:
print(type(response_counter), repr(response_counter), sep='\n')

In [ ]:
response_counter.__closure__

In [ ]:
response_counter.__closure__[0].cell_contents

* One case where closures are frequently used is in building function wrappers
* Suppose we want to log each invocation of a function:

In [ ]:
def logging(func):
    def wrapper(*args, **kwargs):
        print(f'Calling {func}({args}, {kwargs})')
        return func(*args, **kwargs)
        
    return wrapper

In [ ]:
logging_rc = logging(response_counter)

In [ ]:
print(response_counter())

In [ ]:
print(logging_rc())

In [ ]:
logging_rc.__closure__[0].cell_contents

## Decorators
* Wrapper functions are so common, that Python has its own term for it–a *decorator*.
* Why might you want to use a decorator?
  * sometimes you want to modify a function’s behavior without explicitly modifying the function, e.g., pre/post actions, debugging, etc. 
  * suppose we have a set of tasks that need to be performed by many different functions, e.g.,
   * access control
   * cleanup
   * error handling
   * logging
 * ...in other words, there is some boilerplate code that needs to be executed before or after  every invocation of the function


## Decorators build on topics we already know...
* nested functions
* variable positional args (__`*args`__)
* variable keyword args (__`**kwargs`__)
* functions are objects (actually everything in Python is an object)

In [ ]:
def document_it(func):
    # below is a nested, or inner function
    def wrapper(*args, **kwargs):
        print(f'Running function: {func.__name__}')
        print(f'Positional arguments: {args}')
        print(f'Keyword arguments: {kwargs}')
        # here we invoke the function passed in as an argument
        result = func(*args, **kwargs)
        print(f'Result: {result}')
        return result
    
    # document_it() is returning a reference to the inner function (wrapper)
    return wrapper

In [ ]:
def add_things(a, b):
    """Add 2 numbers."""
    return a + b

print('Running plain old add_things()')
print(add_things(13, 5))

In [ ]:
# manual decorator assignment
cooler_add_things = document_it(add_things) 

In [ ]:
#print('Running cooler_add_things()')
cooler_add_things(13, 5)

In [ ]:
# decorator shorthand for what we did above

@document_it
def add_things(a, b):
    """Add two numbers, a and b."""
    return a + b

# add_things = document_it(add_things)

print(add_things(13, -5))

#### Now that we know the above syntax, we will modify our decorator prototype

In [16]:
from functools import wraps
# the wraps decorator makes it so the metadata of the wrapped function is preserved

def document_it(func):
    # below is a nested, or inner function
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f'Running function: {func.__name__}')
        print(f'Positional arguments: {args}')
        print(f'Keyword arguments: {kwargs}')
        # here we invoke the function passed in as an argument
        result = func(*args, **kwargs)
        print(f'Result: {result}')
        return result
    
    # document_it() is returning a reference to the inner function (wrapper)
    return wrapper

In [17]:
@document_it
def test_func(x, y):
    """important func"""
    return x + y

In [20]:
print(test_func.__name__, test_func.__doc__, sep='\n')

wrapper
None


## Lab: Decorators
1. Make a timer decorator that computes the elapsed time of the function wrapped by it
2. Create some function which takes an integer as its parameter
   * create a wrapper that ensures the parameter is positive
   * use that wrapper to decorate your original function
2. Make a "call counter" decorator that keeps track of how many times a function has been called
2. Write your own "cache" decorator like the one in functools–in other words, it stores function results in a dictionary so repeated calls with the same arguments return instantly rather than re-computing a result that has been previously computed